In [13]:
# Exploration mesures de pesticides

In [14]:
import polars as pl
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df = (
    pl.read_excel(
        "../../data/raw/pesticides_2002_2023_v07_2025.xlsx",
        sheet_name="pesticides_2002_2023_v-07-2025",
    )
    .with_columns([
        pl.col("Code INSEE").cast(pl.Utf8).str.zfill(5),
        pl.col("Debut prelevement").cast(pl.Datetime).dt.date(),
        pl.col("Fin prelevement").cast(pl.Datetime).dt.date(),
    ])
)
df.shape

(994257, 17)

In [16]:
df.describe()

statistic,AASQA,Commune,Code INSEE,xlamb93,ylamb93,Debut prelevement,Fin prelevement,Annee,Semaine,Nbre de jours de prelevement,Coupure PM,Debit de prelevement (m3/h),Substance active,Concentration (ng/m3),Detection/quantification,LD (ng/m3),LQ (ng/m3)
str,str,str,str,f64,f64,str,str,f64,f64,f64,str,f64,str,f64,str,str,str
"""count""","""994257""","""994257""","""969548""",994257.0,994257.0,"""994257""","""994257""",994257.0,994257.0,994257.0,"""994257""",946620.0,"""994257""",994257.0,"""994257""","""994257""","""994257"""
"""null_count""","""0""","""0""","""24709""",0.0,0.0,"""0""","""0""",0.0,0.0,0.0,"""0""",47637.0,"""0""",0.0,"""0""","""0""","""0"""
"""mean""",null,null,null,585723.324037,6.6132e6,"""2017-01-04 21:45:29.075882""","""2017-01-11 10:57:27.915780""",2016.494243,27.874376,6.550441,null,2.756342,null,0.26567,null,null,null
"""std""",null,null,null,1.1688e6,583001.590285,null,null,5.503934,11.806437,1.841203,null,6.170097,null,25.725408,null,null,null
"""min""","""AIR BREIZH""","""ARLES""","""02239""",-6.469109e6,483648.0,"""2002-04-08""","""2002-04-10""",2002.0,1.0,1.0,"""PM10""",1.0,"""2,4-D""",0.0,"""""","""""",""""""
"""25%""",null,null,null,486329.0,6.511113e6,"""2014-04-22""","""2014-04-25""",2014.0,19.0,7.0,null,1.0,null,0.0,null,null,null
"""50%""",null,null,null,622785.0,6.704644e6,"""2018-09-17""","""2018-09-24""",2018.0,27.0,7.0,null,1.0,null,0.0,null,null,null
"""75%""",null,null,null,783039.0,6.844846e6,"""2021-05-31""","""2021-06-07""",2021.0,37.0,7.0,null,1.0,null,0.0,null,null,null
"""max""","""QUALITAIR CORSE""","""Ã‰couflant""","""97617""",1.0257384e7,7.092427e6,"""2023-12-26""","""2024-01-02""",2023.0,53.0,68.0,"""TSP""",30.0,"""Zoxamide""",12649.327,"""<LQ""","""5.95""","""8.67"""


In [17]:
# Codes insee
print("Nombre de communes: ", df["Code INSEE"].n_unique())
print("Liste des communes: ")
df["Code INSEE"].unique().sort()

Nombre de communes:  183
Liste des communes: 


Code INSEE
str
null
"""02239"""
"""02691"""
"""02701"""
"""06029"""
…
"""97224"""
"""97226"""
"""97310"""


Les codes postaux marqués comme null sont les codes postaux de la Corse, à traiter plus tard

In [18]:
# Visualisation des communes avec Code INSEE null
from pyproj import Transformer
import folium

# Extraire les lignes avec code INSEE null et dédupliquer par commune
communes_null = (
    df
    .filter(pl.col("Code INSEE").is_null())
    .select(["Commune", "AASQA", "xlamb93", "ylamb93"])
    .unique()
    .sort("Commune")
)

print(f"{communes_null.shape[0]} stations avec Code INSEE null :")
print(communes_null)

# Conversion Lambert 93 → WGS84
transformer = Transformer.from_crs("EPSG:2154", "EPSG:4326", always_xy=True)
x = communes_null["xlamb93"].cast(pl.Float64).to_numpy()
y = communes_null["ylamb93"].cast(pl.Float64).to_numpy()
lon, lat = transformer.transform(x, y)

# Carte centrée sur le barycentre des points
centre_lat = lat.mean()
centre_lon = lon.mean()

m = folium.Map(location=[centre_lat, centre_lon], zoom_start=7, tiles="CartoDB positron")

for i, row in enumerate(communes_null.iter_rows(named=True)):
    folium.CircleMarker(
        location=[lat[i], lon[i]],
        radius=2,
        color="#e63946",
        fill=True,
        fill_color="#e63946",
        fill_opacity=0.8,
        tooltip=f"{row['Commune']} ({row['AASQA']})",
    ).add_to(m)

m

8 stations avec Code INSEE null :
shape: (8, 4)
┌───────────────────┬─────────────────┬─────────┬─────────┐
│ Commune           ┆ AASQA           ┆ xlamb93 ┆ ylamb93 │
│ ---               ┆ ---             ┆ ---     ┆ ---     │
│ str               ┆ str             ┆ i64     ┆ i64     │
╞═══════════════════╪═════════════════╪═════════╪═════════╡
│ Ajaccio           ┆ QUALITAIR CORSE ┆ 1179274 ┆ 6111348 │
│ Ajaccio           ┆ QUALITAIR CORSE ┆ 1177998 ┆ 6111721 │
│ Ajaccio           ┆ QUALITAIR CORSE ┆ 1176382 ┆ 6108865 │
│ AlÃ©ria           ┆ QUALITAIR CORSE ┆ 1235925 ┆ 6133290 │
│ Ghisonaccia       ┆ QUALITAIR CORSE ┆ 1230789 ┆ 6123310 │
│ Lucciana          ┆ QUALITAIR CORSE ┆ 1232169 ┆ 6181396 │
│ Patrimonio        ┆ QUALITAIR CORSE ┆ 1221346 ┆ 6198532 │
│ Sarrola-Carcopino ┆ QUALITAIR CORSE ┆ 1183045 ┆ 6113790 │
└───────────────────┴─────────────────┴─────────┴─────────┘


In [19]:
# Visualisation des communes avec Code Insee null


In [20]:
df.columns

['AASQA',
 'Commune',
 'Code INSEE',
 'xlamb93',
 'ylamb93',
 'Debut prelevement',
 'Fin prelevement',
 'Annee',
 'Semaine',
 'Nbre de jours de prelevement',
 'Coupure PM',
 'Debit de prelevement (m3/h)',
 'Substance active',
 'Concentration (ng/m3)',
 'Detection/quantification',
 'LD (ng/m3)',
 'LQ (ng/m3)']

In [21]:
# Agrégation de tous les pesticides par date de mesure
agg_par_date = (
    df
    .group_by(["xlamb93", "ylamb93", "Debut prelevement", "Fin prelevement"])
    .agg([
        pl.col("Concentration (ng/m3)").sum().alias("concentration_totale_ng_m3"),
        pl.col("Substance active")
          .filter(pl.col("Concentration (ng/m3)") > 0)
          .n_unique()
          .alias("nb_substances_detectees"),
    ])
    .sort(["xlamb93", "Debut prelevement"])
)

agg_par_date

xlamb93,ylamb93,Debut prelevement,Fin prelevement,concentration_totale_ng_m3,nb_substances_detectees
i64,i64,date,date,f64,u32
-6469109,4185780,2018-06-25,2018-07-02,0.093,2
-6469109,4185780,2018-07-16,2018-07-23,0.177,3
-6469109,4185780,2018-08-06,2018-08-13,0.177,5
-6469109,4185780,2018-08-27,2018-09-03,0.399,5
-6469109,4185780,2018-09-17,2018-09-24,0.231,3
…,…,…,…,…,…
10257384,483648,2022-02-09,2022-02-16,2.12,5
10257384,483648,2022-03-01,2022-03-08,1.562,4
10257384,483648,2022-03-08,2022-03-15,1.786,2


In [ ]:

import ipywidgets as widgets
from IPython.display import display
import plotly.graph_objects as go

# --- Données agrégées par station × période ---
agg = (
    df
    .filter(pl.col("Code INSEE").is_not_null())
    .group_by(["Code INSEE", "Commune", "Debut prelevement", "Fin prelevement"])
    .agg([
        pl.col("Concentration (ng/m3)").sum().alias("concentration_totale"),
        pl.col("Substance active")
          .filter(pl.col("Concentration (ng/m3)") > 0)
          .n_unique()
          .alias("nb_substances_detectees"),
    ])
    .sort(["Code INSEE", "Debut prelevement"])
    .to_pandas()
)
agg["Debut prelevement"] = pd.to_datetime(agg["Debut prelevement"])
agg["dep"] = agg["Code INSEE"].str[:2]

# --- Enrichissement région depuis communes_admin ---
communes_admin = pl.read_parquet("../../data/parquet/communes_admin.parquet").to_pandas()
communes_admin["dep"] = communes_admin["code_insee"].str[:2]
dep_to_reg = (
    communes_admin[["dep", "code_insee_reg"]]
    .drop_duplicates("dep")
    .set_index("dep")["code_insee_reg"]
    .to_dict()
)
reg_to_nom = {v: k for k, v in {
    "Guadeloupe": "01", "Martinique": "02", "Guyane": "03", "La Réunion": "04",
    "Mayotte": "06", "Île-de-France": "11", "Centre-Val de Loire": "24",
    "Bourgogne-Franche-Comté": "27", "Normandie": "28", "Hauts-de-France": "32",
    "Grand Est": "44", "Pays de la Loire": "52", "Bretagne": "53",
    "Nouvelle-Aquitaine": "75", "Occitanie": "76", "Auvergne-Rhône-Alpes": "84",
    "Provence-Alpes-Côte d'Azur": "93", "Corse": "94",
}.items()}
agg["region_code"] = agg["dep"].map(dep_to_reg)
agg["region_nom"]  = agg["region_code"].map(reg_to_nom).fillna("Inconnue")

# --- Widgets ---
regions_dispo = sorted(agg["region_nom"].dropna().unique())
w_region = widgets.Dropdown(
    options=["(toutes)"] + regions_dispo,
    description="Région :", style={"description_width": "80px"}, layout={"width": "300px"}
)
w_dep = widgets.Dropdown(
    description="Dépt :", style={"description_width": "80px"}, layout={"width": "220px"}
)
w_station = widgets.Dropdown(
    description="Station :", style={"description_width": "80px"}, layout={"width": "340px"}
)
w_metric = widgets.ToggleButtons(
    options=[("Concentration totale", "concentration_totale"), ("Nb substances détectées", "nb_substances_detectees")],
    style={"description_width": "0px"},
)
out = widgets.Output()

def get_deps(region):
    if region == "(toutes)":
        sub = agg
    else:
        sub = agg[agg["region_nom"] == region]
    deps = sorted(sub["dep"].unique())
    return ["(tous)"] + deps

def get_stations(region, dep):
    sub = agg.copy()
    if region != "(toutes)":
        sub = sub[sub["region_nom"] == region]
    if dep != "(tous)":
        sub = sub[sub["dep"] == dep]
    stations = (
        sub[["Code INSEE", "Commune"]]
        .drop_duplicates()
        .sort_values("Code INSEE")
    )
    return [(f"{r['Code INSEE']} — {r['Commune']}", r["Code INSEE"]) for _, r in stations.iterrows()]

def on_region_change(change):
    deps = get_deps(w_region.value)
    w_dep.options = deps
    w_dep.value   = deps[0]  # déclenche on_dep_change

def on_dep_change(change):
    stations = get_stations(w_region.value, w_dep.value)
    w_station.options = stations
    if stations:
        w_station.value = stations[0][1]
    draw(None)

def draw(change):
    code = w_station.value
    metric = w_metric.value
    data = agg[agg["Code INSEE"] == code].sort_values("Debut prelevement")
    commune = data["Commune"].iloc[0] if len(data) else code

    with out:
        out.clear_output(wait=True)
        if metric == "concentration_totale":
            ylabel = "Concentration totale (ng/m³)"
            hover  = (
                "<b>%{x|%d/%m/%Y}</b><br>"
                "Concentration : %{y:.3f} ng/m³<br>"
                "Substances détectées : %{customdata}<extra></extra>"
            )
            y_vals      = data["concentration_totale"]
            custom_data = data["nb_substances_detectees"]
        else:
            ylabel = "Nb substances détectées"
            hover  = (
                "<b>%{x|%d/%m/%Y}</b><br>"
                "Substances détectées : %{y}<br>"
                "Concentration totale : %{customdata:.3f} ng/m³<extra></extra>"
            )
            y_vals      = data["nb_substances_detectees"]
            custom_data = data["concentration_totale"]

        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=data["Debut prelevement"],
            y=y_vals,
            customdata=custom_data,
            mode="lines+markers",
            line=dict(color="#2196F3", width=1.5),
            marker=dict(size=4),
            fill="tozeroy",
            fillcolor="rgba(33,150,243,0.1)",
            hovertemplate=hover,
        ))
        fig.update_layout(
            title=f"{commune} ({code})",
            yaxis_title=ylabel,
            xaxis_title=None,
            height=380,
            margin=dict(l=50, r=20, t=45, b=40),
            hovermode="x unified",
        )
        fig.show()

# Cascades
w_region.observe(on_region_change, names="value")
w_dep.observe(on_dep_change, names="value")
w_metric.observe(draw, names="value")

# Init
on_region_change(None)

display(widgets.VBox([
    widgets.HBox([w_region, w_dep, w_station]),
    w_metric,
    out,
]))

In [ ]:
# Recupération des données météo pour le même lieu et période, depuis l'API meteofrance


In [23]:
# Conversion des coordonnées de la station en lat/lon pour requêter l'API météo
lat, lon = transformer.transform(x[0], y[0])
print(f"Coordonnées de la station : lat={lat:.6f}, lon={lon:.6f}")  

Coordonnées de la station : lat=8.772569, lon=41.945044


In [24]:
# Recupération des données météo pour la période de mesure
from datetime import datetime
date_debut = agg["Debut prelevement"].min()
date_fin   = agg["Debut prelevement"].max()
print(f"Période de mesure : {date_debut.date()} à {date_fin.date()}")


Période de mesure : 2002-04-08 à 2023-12-26
